# AI Usage Protocol Notebook

**Student:** Md Ratul Ahmed Alvi  
**Course:** HW/SW Codesign — Prof. Dr. Achim Rettberg  
**Milestone:** Seminar presentation  
**Topic:** High-Level Synthesis for Scheduling Pipelined Circuits  
**Protocol version:** 0.2  
**Date:** June 22, 2026


---
## Interaction 1 — Understanding what the Initiation Interval means

**ID:** e-001  
**AI tool:** Claude (claude.ai), Claude Sonnet 4.6

### Objective
The Teich/de Micheli textbook defined II as the number of clock cycles between starting consecutive iterations, but I did not fully understand why this number exists as a concept separate from latency. I wanted to understand it intuitively before designing the presentation structure.

### Prompt I used
```
I am reading Teich and de Micheli on pipeline scheduling. What is the Initiation
Interval (II) and why does it matter for pipelining? How is it different from
latency? Explain it simply, like for a student.
```

### What AI gave me
Claude explained II using an assembly-line analogy: latency is how long one product takes to move through the whole line; II is how often a new product enters the line. It gave the key insight that pipelining improves throughput (1/II) without changing latency. This matched what the textbook said but in language I could follow.

### What I verified
- Checked the analogy against Section on pipeline scheduling in Teich/de Micheli — the factory/assembly line framing matches exactly what the textbook describes conceptually.
- Confirmed throughput = 1/II is stated directly in the textbook.
- **Verification level:** primary-source-check

### How I used it
- I wrote the pipelining definitions slide (slide 3) myself based on understanding from this explanation and the textbook.
- The assembly-line analogy appears in my verbal explanation during the presentation, not on the slides — this came from my own understanding.
- **Used in slides:** Yes — conceptual foundation for slide 3
- **Direct text reused:** No
- **Usage type:** terminology-clarification
- **Status:** accepted

### Reflection
This explanation made me realise that II is essentially a performance metric for the pipeline as a whole, not for any individual computation. Once I understood this, I was able to decide which three experiments would best show the different limits on II — that design decision was my own.


---
## Interaction 2 — Understanding the difference between ResII and RecII

**ID:** e-002  
**AI tool:** Claude (claude.ai), Claude Sonnet 4.6

### Objective
The textbook gives two separate lower bounds on II: one from resources and one from recurrences. I understood the formulas but not the intuitive difference — specifically why adding hardware helps one but not the other.

### Prompt I used
```
In de Micheli the book says II_min = max(ResII, RecII). I understand ResII is
about hardware units and RecII is about loop dependencies. But why can you fix
ResII by adding hardware and not RecII? What does RecII actually mean physically?
```

### What AI gave me
Claude explained that ResII is a queue problem — if three people need to use one printer, they wait in line and adding a second printer solves it. RecII is a chain problem — if person B cannot start until person A finishes and gives them the result, it does not matter how many printers you add, the chain still forces the wait. The physical meaning is that RecII comes from a loop-carried dependency: iteration k+1 reads a value that iteration k must write first.

### What I verified
- Compared against the accumulation loop example in the textbook — the `sum += ...` pattern is the exact case where A1 in iteration k+1 needs the sum produced by A1 in iteration k. This matches the chain analogy.
- Confirmed the formula RecII = ceil(D/L) where D is the delay around the dependency loop and L is the iteration distance — taken directly from de Micheli.
- **Verification level:** primary-source-check

### How I used it
- This understanding directly led me to design slide 4 (Problem Statement) as a two-card layout showing resources vs dependencies separately.
- The printer/chain analogies were my personal study notes — not on the slides. On the slides I wrote the formal explanations in my own words.
- The selection of one FIR experiment (resource-limited), one accumulation experiment (dependency-limited), and one delay-line experiment (neither) was my own decision, based on understanding this distinction.
- **Used in slides:** Yes — conceptual foundation for slides 4, 7, 9
- **Direct text reused:** No
- **Usage type:** terminology-clarification
- **Status:** accepted

### Reflection
The most valuable thing from this interaction was understanding *why* the three experiments cover different cases. I chose one experiment to show ResII dominating, one to show RecII dominating, and one to show neither — that structure is entirely my own design. The AI helped me understand the theory well enough to make that design decision confidently.


---
## Interaction 3 — Understanding modulo scheduling (the cycle mod II check)

**ID:** e-003  
**AI tool:** Claude (claude.ai), Claude Sonnet 4.6

### Objective
The list scheduling algorithm in the textbook checks resource availability at slot (cycle mod II) rather than the absolute cycle number. I could not understand why this works and needed to be able to explain it during the presentation if asked.

### Prompt I used
```
In list scheduling for pipelined circuits the scheduler checks (cycle mod II)
for resource availability instead of the absolute cycle number. Why does this
work? Explain it simply — like I am a student who just needs to understand it,
not implement it.
```

### What AI gave me
Claude explained that every iteration repeats the exact same pattern of operations every II cycles. So cycle 0, cycle 3, cycle 6 (with II=3) all correspond to the same position in the repeating pattern — the scheduler only needs to track one window of II slots because it repeats forever. It compared this to a clock with II hours instead of 12.

### What I verified
- Traced through the FIR example manually: M1 at cycle 0, then at cycle 3, then at cycle 6 — these all fall on slot 0 mod 3. This confirmed the explanation is correct.
- Checked the textbook description of modulo scheduling — it confirms the repeating window concept.
- **Verification level:** methodological-check (manual trace)

### How I used it
- This understanding let me explain the schedule grids in slides 6, 8, 10 with confidence during the presentation.
- It also helped me explain to myself why the three pipeline grids look like staircases — each new iteration starts II columns to the right.
- The concept was not added as a separate slide — it was background knowledge I needed to answer questions about how the scheduler works.
- **Used in slides:** No (background knowledge for Q&A)
- **Direct text reused:** No
- **Usage type:** terminology-clarification
- **Status:** accepted

### Reflection
Being able to explain modulo scheduling is important for the presentation because the pipeline grids are the core visual of slides 6, 8, and 10. Without understanding why each row shifts by exactly II columns, I would not be able to defend those slides in the Q&A. This interaction gave me that understanding — but drawing the grids, deciding which iterations to show, and writing the slide text was all my own work.


---
## Interaction 4 — Understanding why the accumulation loop achieves II=2 and not II=1

**ID:** e-004  
**AI tool:** Claude (claude.ai), Claude Sonnet 4.6

### Objective
The textbook formula gives RecII = ceil(1/1) = 1 for the accumulation loop. But when I worked through the scheduling manually, I could see that II=1 is not achievable because the multiply takes 2 cycles. I needed to understand this properly so I could explain it correctly on slide 7/8.

### Prompt I used
```
For an accumulation loop sum += x[i] * coeff with one multiplier and one adder:
the RecII formula gives ceil(1/1) = 1. But the multiply takes 2 cycles so A1
cannot start until cycle 2. Does this mean II = 1 is not achievable even though
RecII = 1? Why?
```

### What AI gave me
Claude explained that RecII is a theoretical lower bound from the dependency distance alone. The list scheduler must additionally satisfy operation latency constraints. Since M1 has latency 2, ASAP(A1) = 2, and the scheduler cannot construct a valid repeating schedule at II=1 because the operations do not fit in a 1-cycle window. The first feasible schedule is at II=2. This is not a contradiction — it means both RecII and operation latency together constrain the achievable II, and the actual result is the maximum of all constraints.

### What AI got wrong / where it was not enough
The first explanation mixed up two different dependency types (within-iteration dependency M1→A1, and the loop-carried dependency A1→A1 across iterations). I had to ask a follow-up clarification to get a clean explanation that separated these two concepts. The first response would not have been safe to use directly for explaining in the presentation.

### What I verified
- Manually traced the scheduler: tried II=1, confirmed M1 at cycle 0 and A1 at cycle 2 cannot both satisfy the modulo-1 resource window.
- Tried II=2 manually: M1@0, A1@2. Confirmed this works with no conflicts.
- Checked that the textbook states RecII is a lower bound — it does not claim it is always achievable.
- **Verification level:** methodological-check (manual trace)

### How I used it
- Slide 7 caption: "Although the theoretical lower bound is II=1, the first feasible schedule was obtained at II=2" — this phrasing was written by me after understanding the distinction.
- Slide 8 explanation: the visible gaps in the pipeline grid are my own observation from tracing the schedule; the explanation on the slide is mine.
- **Used in slides:** Yes — slides 7 and 8
- **Direct text reused:** No
- **Usage type:** argument-critique
- **Status:** revised (needed follow-up to clarify)

### Reflection
This was the hardest concept in the whole presentation to get right. The AI gave a correct answer but the first version was not precise enough. Asking the follow-up question and then manually tracing the scheduler step by step on paper gave me the real understanding. The explanation on slide 7 is something I can now defend confidently in the Q&A because I worked through it myself.


---
## Interaction 5 — Generating the PPTX slide file

**ID:** e-005  
**AI tool:** Claude (claude.ai), Claude Sonnet 4.6

### Objective
After designing the presentation structure, deciding the slide content, and understanding all concepts, I asked Claude to produce the PPTX file because building a 14-slide formatted presentation in PowerPoint manually would take several hours and the formatting quality matters for a seminar presentation.

### Prompt I used (representative — actual prompts were iterative)
```
Create a 14-slide PPTX presentation for my HLS seminar. White background, no
funky colors, plain and simple. Slides should be:
1. Title: High-Level Synthesis for Scheduling Pipelined Circuits
2. Introduction to HLS (flow: C/C++ → Scheduling → Allocation → Binding → RTL)
3. Pipelining definitions (Latency, Throughput, II, ResII, RecII, formula)
4. Problem statement (why cannot all designs achieve II=1, ResII vs RecII cards)
5-6. Experiment 1: FIR filter (DFG + schedule grid showing M1 M2 M3 A1 A2 +
     pipeline with 3 iterations)
7-8. Experiment 2: Accumulation (DFG with loop-carried dependency + schedule
     showing M1 starts / M1 busy / A1 + pipeline with 4 iterations)
9-10. Experiment 3: Delay line (DFG + pipeline showing 2 multipliers alternating)
11. C++ terminal output
12. Comparison table (all 3 experiments)
13. Conclusion
14. Questions slide (black background, large text)
Add page numbers bottom-right. Add references bottom-left on slides that use
Teich/de Micheli or Sllame 2003.
```

### What AI gave me
Claude generated a pptxgenjs Node.js script that produced the PPTX file. It went through 3 iterations:
- **Version 1:** Produced working slides but the pipeline grid for the FIR filter showed 'M1, M1, M2' (repeating M1 twice to show latency) — this was visually wrong. The correct grid shows M1, M2, M3 in consecutive cycles because the multiplier is pipelined.
- **Version 2:** Fixed the FIR grid. But the title was wrong and the VHDL placeholder slide was included.
- **Version 3 (final):** Title changed to correct title, VHDL slide removed, Questions slide added as separate page.

### What AI got wrong
- The pipeline grid for the FIR filter was initially incorrect — showed M1 twice instead of M1, M2, M3. I identified this error by comparing the grid against my manually computed schedule.
- Early versions included unnecessary visual elements that I asked to be removed.
- The PPTX formatting does not perfectly match what you see in Keynote/LibreOffice vs PowerPoint — some fonts and spacing shift. Minor cosmetic issue.

### What I verified
- Opened every generated PPTX in LibreOffice and checked each slide against my intended content.
- Compared pipeline grids on slides 6, 8, 10 against manually computed schedules.
- Confirmed all reference citations on the bottom of slides match the actual sources.
- Confirmed the formulas on slide 3 match the textbook exactly.
- **Verification level:** empirical-check (opened and reviewed each slide)

### How I used it
- The final PPTX file (HLS_Simple_Final_v3.pptx) is the presentation I submitted and presented.
- All content on every slide was determined by me — the text, the experiments, the schedule values, the conclusions. Claude formatted it.
- **Used in slides:** Yes — entire presentation file
- **Direct text reused:** The slide text reflects my own understanding. Formulas and citations came from the textbook.
- **Usage type:** other (presentation formatting/production)
- **Status:** revised (3 iterations to correct errors)

### Reflection
Using AI to produce the PPTX file saved significant time on formatting. However, it required me to review every slide carefully because the first version had a factual error in the pipeline grid that would have been wrong in the presentation. The content — which experiments to include, what the pipeline grids should show, what the conclusion says — was all decided by me before I gave the prompt. The AI was a formatting tool, not a content generator for this interaction.


In [ ]:
import json, re
from pathlib import Path
from datetime import datetime

PROTOCOL_VERSION = "0.2"

protocol_data = {
    "protocol_version": PROTOCOL_VERSION,
    "student_id": "Md Ratul Ahmed Alvi",
    "course": "HW/SW Codesign (Prof. Dr. Achim Rettberg)",
    "milestone": "seminar-presentation",
    "topic": "High-Level Synthesis for Scheduling Pipelined Circuits",
    "exported_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "entries": [
        {
            "id": "e-001",
            "timestamp": "2026-06-10T14:00:00+02:00",
            "parent_id": None,
            "objective": "Understand the concept of Initiation Interval (II) and how it differs from latency, before designing the presentation structure.",
            "ai_tool": {"name": "Claude", "model": "Claude Sonnet 4.6 (claude.ai)"},
            "prompt": "I am reading Teich and de Micheli on pipeline scheduling. What is the Initiation Interval (II) and why does it matter for pipelining? How is it different from latency? Explain it simply, like for a student.",
            "ai_output_summary": "Claude explained II using an assembly-line analogy: latency is how long one product takes; II is how often a new product enters the line. Gave key insight that throughput = 1/II while latency is unchanged by pipelining. Matched textbook description.",
            "verification": {
                "actions": [
                    "Checked analogy against Teich/de Micheli pipeline scheduling section — framing matches.",
                    "Confirmed throughput = 1/II is stated directly in the textbook."
                ],
                "level": "primary-source-check"
            },
            "evaluation": {
                "status": "accepted",
                "issues": [],
                "usefulness": "high"
            },
            "integration": {
                "used_in_work": True,
                "section": "Slide 3 — Pipelining in HLS (conceptual foundation)",
                "usage_type": "terminology-clarification",
                "direct_text_reused": False
            },
            "reflection": "This explanation gave me the conceptual foundation to decide which three experiments would best illustrate different limits on II. That design decision — one resource-limited, one dependency-limited, one neither — was my own."
        },
        {
            "id": "e-002",
            "timestamp": "2026-06-11T10:00:00+02:00",
            "parent_id": None,
            "objective": "Understand intuitively why ResII can be reduced by adding hardware but RecII cannot, before writing the problem statement slide.",
            "ai_tool": {"name": "Claude", "model": "Claude Sonnet 4.6 (claude.ai)"},
            "prompt": "In de Micheli the book says II_min = max(ResII, RecII). I understand ResII is about hardware units and RecII is about loop dependencies. But why can you fix ResII by adding hardware and not RecII? What does RecII actually mean physically?",
            "ai_output_summary": "Claude explained ResII as a queue problem (adding printers solves a printing queue) and RecII as a chain problem (B cannot start until A hands over the result — more printers do not help). The loop-carried dependency in sum += ... is the chain: iteration k+1 needs the sum from iteration k.",
            "verification": {
                "actions": [
                    "Compared against accumulation loop example in textbook — chain analogy matches the dependency description exactly.",
                    "Confirmed RecII = ceil(D/L) formula is from de Micheli, not from the AI."
                ],
                "level": "primary-source-check"
            },
            "evaluation": {
                "status": "accepted",
                "issues": [],
                "usefulness": "high"
            },
            "integration": {
                "used_in_work": True,
                "section": "Slide 4 (Problem Statement) and slides 7, 9 — experiment bottleneck explanations",
                "usage_type": "terminology-clarification",
                "direct_text_reused": False
            },
            "reflection": "Understanding this distinction allowed me to structure the three experiments deliberately: FIR (ResII dominates), accumulation (RecII dominates), delay line (neither). The experiment selection was my own decision."
        },
        {
            "id": "e-003",
            "timestamp": "2026-06-12T15:00:00+02:00",
            "parent_id": None,
            "objective": "Understand why the list scheduler checks (cycle mod II) for resource availability, to be able to explain the pipeline grids in the presentation Q&A.",
            "ai_tool": {"name": "Claude", "model": "Claude Sonnet 4.6 (claude.ai)"},
            "prompt": "In list scheduling for pipelined circuits the scheduler checks (cycle mod II) for resource availability instead of the absolute cycle number. Why does this work? Explain it simply — like I am a student who just needs to understand it, not implement it.",
            "ai_output_summary": "Claude explained that every pipeline iteration repeats the same operation pattern every II cycles. Cycles 0, 3, 6 with II=3 all map to the same slot — like a clock with II positions. The scheduler only needs to track one window of II slots because the pattern repeats forever.",
            "verification": {
                "actions": [
                    "Traced FIR example manually: M1 at cycles 0, 3, 6 all fall on slot 0 mod 3. Confirmed.",
                    "Checked modulo scheduling concept in textbook — confirms the repeating window idea."
                ],
                "level": "methodological-check"
            },
            "evaluation": {
                "status": "accepted",
                "issues": [],
                "usefulness": "medium"
            },
            "integration": {
                "used_in_work": False,
                "section": "Background knowledge for Q&A on pipeline grid slides (6, 8, 10)",
                "usage_type": "terminology-clarification",
                "direct_text_reused": False
            },
            "reflection": "This interaction was purely for self-understanding. The pipeline grids in slides 6, 8, 10 are my own work based on manually computing the schedules — the modulo concept explains why the staircase pattern shifts exactly II columns between rows."
        },
        {
            "id": "e-004",
            "timestamp": "2026-06-13T11:00:00+02:00",
            "parent_id": None,
            "objective": "Resolve why the accumulation loop achieves II=2 even though the RecII formula gives 1, so I can explain slide 7 correctly.",
            "ai_tool": {"name": "Claude", "model": "Claude Sonnet 4.6 (claude.ai)"},
            "prompt": "For accumulation loop sum += x[i] * coeff with one multiplier and one adder: the RecII formula gives ceil(1/1) = 1. But the multiply takes 2 cycles so A1 cannot start until cycle 2. Does this mean II=1 is not achievable even though RecII=1? Why?",
            "ai_output_summary": "Claude explained RecII is a theoretical lower bound from dependency distance only. The scheduler also enforces operation latency constraints. Since M1 has latency 2, ASAP(A1)=2, and a 1-cycle repeating window cannot accommodate both M1 and A1 — so II=1 is infeasible. First feasible schedule is II=2. First explanation mixed up intra-iteration and loop-carried dependency — required a follow-up to clarify.",
            "verification": {
                "actions": [
                    "Manually traced scheduler: tried II=1 — M1 latency 2 cannot satisfy modulo-1 window. Confirmed infeasible.",
                    "Tried II=2 manually: M1@0, A1@2. Confirmed this satisfies all constraints.",
                    "Checked textbook: RecII is stated as a lower bound, not a guaranteed achievable value."
                ],
                "level": "methodological-check"
            },
            "evaluation": {
                "status": "revised",
                "issues": [
                    "First AI response mixed up within-iteration dependency (M1 to A1) with loop-carried dependency (A1 to next A1). Required a follow-up question to get a clean explanation."
                ],
                "usefulness": "high"
            },
            "integration": {
                "used_in_work": True,
                "section": "Slides 7–8 (Experiment 2: Accumulation Loop)",
                "usage_type": "argument-critique",
                "direct_text_reused": False
            },
            "reflection": "The AI's first answer was not precise enough and I had to ask a follow-up. Manually tracing the scheduler step by step gave me the actual understanding. The slide caption 'first feasible schedule was obtained at II=2' is phrased by me after working through this carefully."
        },
        {
            "id": "e-005",
            "timestamp": "2026-06-21T14:00:00+02:00",
            "parent_id": None,
            "objective": "Generate the PPTX file for the seminar presentation. All slide content, structure, and experiment design had been determined before this step.",
            "ai_tool": {"name": "Claude", "model": "Claude Sonnet 4.6 (claude.ai)"},
            "prompt": "Create a 14-slide PPTX for my HLS seminar. White background, plain and simple, no funky colors. Title: High-Level Synthesis for Scheduling Pipelined Circuits. Slides: 1 Title, 2 HLS intro with flow diagram, 3 Pipelining definitions, 4 Problem statement with ResII vs RecII, 5-6 Experiment 1 FIR filter with DFG and pipeline grid showing M1 M2 M3 A1 A2 across 3 iterations, 7-8 Experiment 2 Accumulation with DFG loop-carried dependency and schedule showing M1-starts M1-busy A1 gaps, 9-10 Experiment 3 Delay line with 2 multipliers alternating, 11 C++ terminal output, 12 Comparison table, 13 Conclusion, 14 Questions black slide. Add page numbers bottom-right. Add references bottom-left for Teich/de Micheli and Sllame 2003.",
            "ai_output_summary": "Claude generated a pptxgenjs Node.js script producing the PPTX. Required 3 iterations: v1 had an error in the FIR pipeline grid showing M1 twice instead of M1/M2/M3; v2 fixed the grid but had wrong title and included an unwanted VHDL slide; v3 corrected the title to the proper seminar title, removed VHDL slide, and added a standalone Questions slide.",
            "verification": {
                "actions": [
                    "Opened every version of the PPTX in LibreOffice and reviewed all 14 slides.",
                    "Compared pipeline grids on slides 6, 8, 10 against manually computed schedules — v1 grid was wrong (M1 repeated), corrected in v3.",
                    "Verified all formulas on slide 3 match the textbook exactly.",
                    "Confirmed reference citations on slide footers match the actual publications."
                ],
                "level": "empirical-check"
            },
            "evaluation": {
                "status": "revised",
                "issues": [
                    "v1 pipeline grid for FIR showed M1 twice (cycles 0 and 1) — incorrect. Correct is M1 at 0, M2 at 1, M3 at 2.",
                    "v2 had wrong presentation title.",
                    "3 iterations were needed to get a clean final file."
                ],
                "usefulness": "high"
            },
            "integration": {
                "used_in_work": True,
                "section": "Entire presentation (HLS_Simple_Final_v3.pptx)",
                "usage_type": "other",
                "direct_text_reused": False
            },
            "reflection": "The AI was used as a formatting tool here, not a content generator. All content — which slides to include, what experiments to show, what conclusions to draw, what the pipeline grids should contain — was decided by me before I wrote the prompt. The first version had a factual error in the FIR grid that I caught by checking against my manually computed schedule. This required a correction before the file was usable."
        }
    ]
}

def safe_key(t):
    return re.sub(r'[^a-z0-9]+', '-', (t or '').lower()).strip('-') or 'ai-protocol'

def esc(v):
    return str(v or '').replace('\\','\\textbackslash{}').replace('{','\\{').replace('}','\\}').replace('&','\\&').replace('%','\\%').replace('$','\\$').replace('#','\\#').replace('_','\\_')

json_file = "ai_usage_protocol_alvi_presentation.json"
bib_file  = "ai_usage_protocol_alvi_presentation.bib"

Path(json_file).write_text(json.dumps(protocol_data, indent=2, ensure_ascii=False), encoding='utf-8')

year = protocol_data['exported_at'][:4]
bib_key = '-'.join(['ai-usage-protocol', safe_key(protocol_data['student_id']),
                    safe_key(protocol_data['milestone']), year])
title = f"AI Usage Protocol for {protocol_data['milestone']}: {protocol_data['topic']}"
note  = (f"Machine-readable documentation of AI-assisted scientific work; "
         f"course: {protocol_data['course']}; protocol version: {protocol_data['protocol_version']}; "
         f"documented reasoning episodes: {len(protocol_data['entries'])}; "
         f"JSON file: {json_file}")

bib = f"""@misc{{{bib_key},
  author = {{{esc(protocol_data['student_id'])}}},
  title = {{{esc(title)}}},
  year = {{{year}}},
  howpublished = {{{esc('Submitted AI usage protocol notebook and JSON file')}}},
  note = {{{esc(note)}}}
}}"""

Path(bib_file).write_text(bib + '\n', encoding='utf-8')

print(f"Saved: {json_file}")
print(f"Saved: {bib_file}")
print(f"Entries: {len(protocol_data['entries'])}")
print()
print('BibTeX entry:')
print(bib)


---
## Controlled Vocabulary

### Verification level
- `primary-source-check` — checked against original textbook (Teich/de Micheli)
- `methodological-check` — checked by working through the reasoning manually (e.g. tracing a schedule by hand)
- `empirical-check` — checked by opening and reviewing the generated output (e.g. PPTX slides)

### Evaluation status
- `accepted` — AI explanation was correct and matched the textbook
- `revised` — AI output had an error or imprecision that required a follow-up correction before use
